In [7]:
import pandas as pd
# For text preprocessing
import nltk
from nltk.corpus import stopwords 
from nltk. tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora 
from gensim.models import LdaModel

# DownLoad NLTK Resources
nltk.download(' stopwords')
nltk.download(' punkt')
nltk. download('wordnet')

[nltk_data] Error loading  stopwords: Package ' stopwords' not found
[nltk_data]     in index
[nltk_data] Error loading  punkt: Package ' punkt' not found in index
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [8]:
documents = {
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
}

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token.isalnum()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

processed_docs = [preprocess(doc) for doc in documents]
processed_docs

[['rafael', 'nadal', 'australian', 'open'],
 ['biden', 'virus', 'plan', 'meet', 'reality'],
 ['biden', 'virus', 'plan', 'stand'],
 ['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open'],
 ['biden', 'announces', 'virus', 'measure']]

In [10]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(processed_docs)

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

In [11]:
# corpus: bag-of-words representation of the documents
# num_topics: number of topics to be extracted by the model
# id2word=dictionary: dictionary mapping from word IDs to words
# passes: number of passes through the corpus during training

# Train an LDA model on the corpus with 2 topics using Gensim's LdaModel class
lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=15)

In [13]:
# empty list to store dominant topic labels for each document
article_labels = []

# iterate over each processed document
for i, doc in enumerate(processed_docs):
    # for each document, convert to bow representation
    bow = dictionary.doc2bow(doc)
    # get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # append to the list
    article_labels.append(dominant_topic)


In [15]:
# Create DataFrame
df = pd.DataFrame({"Article": list(documents), "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df)
print()

Table with Articles and Topic:
                                             Article  Topic
0         Rafael Nadal Is Out of the Australian Open      0
1                   Biden's Virus Plans Meet Reality      1
2                    Where Biden's Virus Plan Stands      1
3  Rafael Nadal Joins Roger Federer in Missing U....      0
4                     Biden Announces Virus Measures      1



In [16]:
print("Top terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}: ")
    terms = [terms.strip() for terms in topic.split("+")]
    for term in terms:
        wight, word = term.split("*")
        print(f"  - {word.strip()} (weight: {wight.strip()})")
    print()

Top terms for Each Topic:
Topic 0: 
  - "open" (weight: 0.127)
  - "nadal" (weight: 0.127)
  - "rafael" (weight: 0.127)
  - "australian" (weight: 0.076)
  - "roger" (weight: 0.076)
  - "join" (weight: 0.076)
  - "federer" (weight: 0.076)
  - "missing" (weight: 0.076)
  - "measure" (weight: 0.038)
  - "announces" (weight: 0.038)

Topic 1: 
  - "biden" (weight: 0.168)
  - "virus" (weight: 0.168)
  - "plan" (weight: 0.122)
  - "meet" (weight: 0.073)
  - "reality" (weight: 0.073)
  - "stand" (weight: 0.073)
  - "announces" (weight: 0.062)
  - "measure" (weight: 0.062)
  - "missing" (weight: 0.025)
  - "federer" (weight: 0.025)

